In [1]:
!rm -rf /root/.cache/huggingface

!pip install -q transformers datasets evaluate scikit-learn pandas numpy matplotlib seaborn
!pip install -q accelerate
!pip install -q sentencepiece
!pip install -q huggingface_hub

In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

In [3]:
!pip install -q seqeval

In [4]:
import re
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, roc_curve, auc
)
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForTokenClassification,
    AutoModelForSequenceClassification, TrainingArguments, Trainer,
    DataCollatorForTokenClassification, DataCollatorWithPadding
)
import evaluate

# ============================
# C. Load Datasets
# ============================
TRAIN_CSV = '/content/Gemini 3.5-Flash_Extended_Synthetic_Dataset.csv'
TEST_CSV  = '/content/Test.csv'

df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print(f"Training samples: {len(df_train)}")
print(f"Test samples:     {len(df_test)}")

# ============================
# D. Parse BIO Tags -> Tokens & NER Labels
# ============================
def parse_bio_tags(bio_text):
    tokens, labels = [], []
    for part in bio_text.split():
        match = re.match(r'^(.*?)(?:[,\s।]*)-(B|I)$', part)
        if match:
            token, tag = match.group(1), match.group(2)
            labels.append('B-Product' if tag == 'B' else 'I-Product')
        else:
            token = part
            labels.append('O')
        tokens.append(token)
    return tokens, labels

df_train['tokens']   = df_train['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[0])
df_train['ner_tags'] = df_train['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[1])

if 'bio_tagged_review' in df_test.columns:
    df_test['tokens']   = df_test['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[0])
    df_test['ner_tags'] = df_test['bio_tagged_review'].apply(lambda x: parse_bio_tags(x)[1])
else:
    df_test['tokens'] = df_test['review'].apply(lambda x: x.split())

# ============================
# E. Label Mappings
# ============================
label_list = ['O', 'B-Product', 'I-Product']
id2label   = {i: label for i, label in enumerate(label_list)}
label2id   = {label: i for i, label in enumerate(label_list)}
num_labels = len(label_list)

# ============================
# F. Split Training Data (80/20)
# ============================
X = df_train[['id', 'review', 'tokens', 'ner_tags']]
y = df_train['sentiment']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)},  Validation size: {len(X_val)}")
X_train = X_train.reset_index(drop=True)
X_val   = X_val.reset_index(drop=True)

# ============================
# G. NER Tokenizer & Alignment (XLM-RoBERTa base)
# ============================
# Use XLM-RoBERTa base for both NER and sentiment.
# We keep the same _safe_from_pretrained helper; just change the model name.
MODEL_NAME = "xlm-roberta-base"

def _safe_from_pretrained(factory, identifier, local_env_var='LOCAL_MODEL_DIR', **kwargs):
    """Try to load a resource from Hugging Face; on failure try a local folder."""
    local_dir = os.environ.get(local_env_var)
    if not local_dir:
        local_dir = os.path.join(os.getcwd(), 'local_model')
    try:
        return factory(identifier, **kwargs)
    except Exception as remote_err:
        import warnings
        warnings.warn(f"Remote load failed ({remote_err}); trying local path: {local_dir}")
        if os.path.isdir(local_dir):
            try:
                return factory(local_dir, local_files_only=True, **kwargs)
            except Exception as local_err:
                raise RuntimeError(
                    f"Failed to load '{identifier}' remotely and from local path '{local_dir}': {local_err}"
                ) from local_err
        else:
            auto_dl = os.environ.get('AUTO_DOWNLOAD_MODEL') == '1'
            if auto_dl:
                try:
                    from huggingface_hub import snapshot_download
                except Exception as imp_err:
                    raise RuntimeError(
                        f"Auto-download requested but huggingface_hub is not available: {imp_err}.\n"
                        f"Install it with `pip install huggingface_hub` or download the model manually.\n"
                        f"Local path '{local_dir}' does not exist and remote load failed: {remote_err}"
                    ) from imp_err
                try:
                    os.makedirs(local_dir, exist_ok=True)
                    print(f"Auto-downloading '{identifier}' into {local_dir} ...")
                    snapshot_download(
                        repo_id=identifier,
                        local_dir=local_dir,
                        local_dir_use_symlinks=False
                    )
                    return factory(local_dir, local_files_only=True, **kwargs)
                except Exception as dl_err:
                    raise RuntimeError(
                        f"Auto-download attempted but failed: {dl_err}\n"
                        f"Remote error: {remote_err}"
                    ) from dl_err

            raise RuntimeError(
                f"Failed to load '{identifier}' remotely: {remote_err}.\n"
                f"Local path '{local_dir}' does not exist. To fix, either ensure network access,\n"
                f"set environment variable LOCAL_MODEL_DIR to a downloaded model folder,\n"
                f"or download the model files into '{local_dir}'.\n"
                f"If you want the script to try downloading automatically, set AUTO_DOWNLOAD_MODEL=1\n"
                f"and install huggingface_hub (`pip install huggingface_hub`)."
            ) from remote_err

# NER tokenizer and model (XLM-RoBERTa)
tokenizer_ner = _safe_from_pretrained(AutoTokenizer.from_pretrained, MODEL_NAME)

def tokenize_and_align_labels(examples, tokenizer, label2id):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        padding=False
    )
    all_labels = []
    for i, label_seq in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label_seq[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        all_labels.append(label_ids)
    tokenized_inputs['labels'] = all_labels
    return tokenized_inputs

def create_ner_dataset(df, tokenizer, label2id):
    tokenized = tokenize_and_align_labels(
        {'tokens': df['tokens'].tolist(), 'ner_tags': df['ner_tags'].tolist()},
        tokenizer, label2id
    )
    return Dataset.from_dict({
        'input_ids':      tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels':         tokenized['labels'],
        'tokens':         df['tokens'].tolist(),
        'ner_tags':       df['ner_tags'].tolist(),
        'id':             df['id'].tolist() if 'id' in df else list(range(len(df)))
    })

train_dataset_ner = create_ner_dataset(X_train, tokenizer_ner, label2id)
val_dataset_ner   = create_ner_dataset(X_val,   tokenizer_ner, label2id)

if 'tokens' in df_test.columns and 'ner_tags' in df_test.columns:
    test_dataset_ner = create_ner_dataset(df_test, tokenizer_ner, label2id)
else:
    test_dataset_ner = None

print(f"NER datasets created -- train: {len(train_dataset_ner)}, val: {len(val_dataset_ner)}")

# ============================
# H. Train Final NER Model (XLM-RoBERTa Token Classification)
# ============================
print("\n=== Training NER model on full training data ===")

full_train = pd.concat([X_train, X_val], ignore_index=True)
full_train['sentiment'] = pd.concat([y_train, y_val], ignore_index=True)
full_train_ds = create_ner_dataset(full_train, tokenizer_ner, label2id)

# Build model from XLM-RoBERTa pretrained weights
config = _safe_from_pretrained(AutoConfig.from_pretrained, MODEL_NAME)
config.num_labels = num_labels
config.id2label   = id2label
config.label2id   = label2id
model_ner = _safe_from_pretrained(
    AutoModelForTokenClassification.from_pretrained,
    MODEL_NAME,
    config=config
)

args_ner = TrainingArguments(
    output_dir="ner_final",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
)

data_collator_ner = DataCollatorForTokenClassification(tokenizer_ner)
metric_ner = evaluate.load("seqeval")

def compute_metrics_ner(eval_preds):
    pred_logits, labels = eval_preds
    pred_logits = np.argmax(pred_logits, axis=2)
    predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(pred_logits, labels)
    ]
    references = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(pred_logits, labels)
    ]
    results = metric_ner.compute(predictions=predictions, references=references)
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

trainer_ner = Trainer(
    model=model_ner,
    args=args_ner,
    train_dataset=full_train_ds,
    eval_dataset=val_dataset_ner,
    data_collator=data_collator_ner,
    processing_class=tokenizer_ner,
    compute_metrics=compute_metrics_ner
)

trainer_ner.train()

# Evaluate on test set if available
if test_dataset_ner is not None:
    test_results_ner = trainer_ner.evaluate(test_dataset_ner)
    print("\n=== NER Test Results ===")
    print(test_results_ner)

model_ner.save_pretrained("ner_model_final")
tokenizer_ner.save_pretrained("ner_tokenizer")
print("\nNER model saved.")

# ============================
# I. Sentiment Analysis - Tokenize & Prepare Datasets (XLM-RoBERTa)
# ============================
# Reuse the same tokenizer (XLM-RoBERTa) for sequence classification.
sa_tokenizer = tokenizer_ner   # same model, same tokenizer

def tokenize_sa(reviews, tokenizer):
    return tokenizer(reviews, truncation=True, padding=False)

X_train_sa      = X_train['review'].tolist()
y_train_sa      = y_train.tolist()
X_val_sa        = X_val['review'].tolist()
y_val_sa        = y_val.tolist()
full_train_sa   = full_train['review'].tolist()
full_train_sa_y = full_train['sentiment'].tolist()

train_enc_sa      = tokenize_sa(X_train_sa,    sa_tokenizer)
val_enc_sa        = tokenize_sa(X_val_sa,      sa_tokenizer)
full_train_enc_sa = tokenize_sa(full_train_sa,  sa_tokenizer)

train_sa_dataset = Dataset.from_dict({
    'input_ids':      train_enc_sa['input_ids'],
    'attention_mask': train_enc_sa['attention_mask'],
    'labels':         y_train_sa,
})
val_sa_dataset = Dataset.from_dict({
    'input_ids':      val_enc_sa['input_ids'],
    'attention_mask': val_enc_sa['attention_mask'],
    'labels':         y_val_sa,
})
full_train_sa_dataset = Dataset.from_dict({
    'input_ids':      full_train_enc_sa['input_ids'],
    'attention_mask': full_train_enc_sa['attention_mask'],
    'labels':         full_train_sa_y,
})

print(f"SA datasets created -- train: {len(train_sa_dataset)}, val: {len(val_sa_dataset)}")

# ============================
# J. Train Sentiment Analysis Model (XLM-RoBERTa Sequence Classification)
# ============================
print("\n=== Training final SA model on full training data ===")

model_sa = _safe_from_pretrained(
    AutoModelForSequenceClassification.from_pretrained,
    MODEL_NAME,
    num_labels=3,
    id2label={0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"},
    label2id={"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}
)

args_sa = TrainingArguments(
    output_dir="sa_final",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
)

data_collator_sa = DataCollatorWithPadding(sa_tokenizer)

def compute_metrics_sa_final(eval_pred):
    preds  = eval_pred.predictions.argmax(-1)
    labels = eval_pred.label_ids
    return {
        'precision': precision_score(labels, preds, average='weighted'),
        'recall':    recall_score(labels, preds, average='weighted'),
        'f1':        f1_score(labels, preds, average='weighted'),
        'accuracy':  accuracy_score(labels, preds)
    }

trainer_sa = Trainer(
    model=model_sa,
    args=args_sa,
    train_dataset=full_train_sa_dataset,
    eval_dataset=val_sa_dataset,
    processing_class=sa_tokenizer,
    data_collator=data_collator_sa,
    compute_metrics=compute_metrics_sa_final
)

trainer_sa.train()

# ============================
# K. Evaluate SA on Test Set
# ============================
test_reviews  = df_test['review'].tolist()
test_enc_sa   = tokenize_sa(test_reviews, sa_tokenizer)

if 'sentiment' in df_test.columns:
    test_labels = df_test['sentiment'].tolist()
else:
    test_labels = [0] * len(df_test)
    print("Warning: No sentiment column in test set - using dummy labels.")

test_sa_dataset = Dataset.from_dict({
    'input_ids':      test_enc_sa['input_ids'],
    'attention_mask': test_enc_sa['attention_mask'],
    'labels':         test_labels
})

test_results_sa = trainer_sa.evaluate(test_sa_dataset)
print("\n=== SA Test Results ===")
print(test_results_sa)

model_sa.save_pretrained("sa_model_final")
sa_tokenizer.save_pretrained("sa_tokenizer")
print("SA model saved.")

# ============================
# L. Inference Example (with device fix)
# ============================
from transformers import pipeline

# NER pipeline using the fine-tuned XLM-RoBERTa model
ner_pipe = pipeline(
    "ner",
    model="ner_model_final",
    tokenizer="ner_tokenizer",
    aggregation_strategy="simple"
)

sample_review = "ফেস ওয়াশ টা অনেক সুন্দর, অসংখ্য ধন্যবাদ ডেলিভারি খুব দ্রুত দেয়ার জন্য।"
print("\nSample review:", sample_review)
print("NER results:", ner_pipe(sample_review))

# Sentiment inference with device handling
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_sa.to(device)

enc = sa_tokenizer([sample_review], padding=True, truncation=True, return_tensors="pt")
enc = {k: v.to(device) for k, v in enc.items()}

with torch.no_grad():
    logits = model_sa(**enc).logits
    pred = logits.argmax(dim=1).item()

sentiment_map = {0: "Negative", 1: "Neutral", 2: "Positive"}
print(f"Sentiment: {sentiment_map[pred]}")

print("\nAll tasks completed successfully!")

Training samples: 3000
Test samples:     30
Train size: 2400,  Validation size: 600


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...


=== SA Test Results ===
{'eval_loss': 1.740541934967041, 'eval_precision': 0.7632478632478633, 'eval_recall': 0.7, 'eval_f1': 0.7066666666666667, 'eval_accuracy': 0.7}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SA model saved.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Sample review: ফেস ওয়াশ টা অনেক সুন্দর, অসংখ্য ধন্যবাদ ডেলিভারি খুব দ্রুত দেয়ার জন্য।
NER results: [{'entity_group': 'Product', 'score': np.float32(0.999907), 'word': 'ফ', 'start': 0, 'end': 1}, {'entity_group': 'Product', 'score': np.float32(0.99973935), 'word': 'েস ওয়াশ', 'start': 1, 'end': 9}]
Sentiment: Positive

All tasks completed successfully!
